# Phase 10 — SQL Validation & Security

This notebook implements a security layer between the LLM-generated SQL
and the query execution engine.

The validator ensures that generated SQL:

- is not empty
- contains only one statement
- is read-only
- uses SELECT queries
- does not contain dangerous SQL commands
- accesses only approved tables
- does not contain comments
- does not exceed the maximum query length
- contains a reasonable LIMIT
- does not access filesystem or external resources

No SQL generated by an LLM should be executed without passing validation.

In [0]:
import re
from typing import Dict, List

In [0]:
# ============================================================
# SQL VALIDATION CONFIGURATION
# ============================================================

MAX_SQL_LENGTH = 5000
MAX_RESULT_ROWS = 1000

ALLOWED_TABLES = {
    "genai_copilot.gold.monthly_sales",
    "genai_copilot.gold.region_sales",
    "genai_copilot.gold.customer_metrics",
    "genai_copilot.gold.product_metrics",
    "genai_copilot.gold.category_metrics",
}

print("SQL validator configuration loaded.")
print("Maximum SQL length:", MAX_SQL_LENGTH)
print("Maximum result rows:", MAX_RESULT_ROWS)
print("Allowed tables:")

for table in sorted(ALLOWED_TABLES):
    print(" -", table)

In [0]:
# ============================================================
# FORBIDDEN SQL OPERATIONS
# ============================================================

FORBIDDEN_KEYWORDS = [
    "DROP",
    "DELETE",
    "UPDATE",
    "INSERT",
    "ALTER",
    "TRUNCATE",
    "MERGE",
    "GRANT",
    "REVOKE",
    "CREATE",
    "REPLACE",
    "COPY",
    "OPTIMIZE",
    "VACUUM",
    "MSCK",
    "SET",
    "RESET",
    "USE",
]

print("Forbidden operations configured:")
print(", ".join(FORBIDDEN_KEYWORDS))

In [0]:
# ============================================================
# FORBIDDEN FUNCTIONS / ACCESS PATTERNS
# ============================================================

FORBIDDEN_PATTERNS = [
    r"\bREAD_FILES\s*\(",
    r"\bREAD_FILE\s*\(",
    r"\bINPUT_FILE_NAME\s*\(",
    r"\bHTTP_REQUEST\s*\(",
    r"\bURL\s*\(",
    r"\bJDBC\s*\(",
    r"\bPYTHON\s*\(",
    r"\bR\s*\(",
    r"\bJAVA\s*\(",
]

print("Forbidden access patterns configured.")

In [0]:
def normalize_sql(sql: str) -> str:
    """
    Normalize SQL for validation.

    This does NOT attempt to rewrite SQL.
    It only removes unnecessary surrounding whitespace.
    """

    if sql is None:
        return ""

    return sql.strip()

In [0]:
def contains_sql_comments(sql: str) -> bool:
    """
    Detect common SQL comment syntax.
    """

    if "--" in sql:
        return True

    if "/*" in sql or "*/" in sql:
        return True

    return False

In [0]:
def contains_multiple_statements(sql: str) -> bool:
    """
    Detect multiple SQL statements.

    We allow one optional trailing semicolon.
    """

    sql = sql.strip()

    if not sql:
        return False

    # Remove one trailing semicolon.
    if sql.endswith(";"):
        sql = sql[:-1].strip()

    return ";" in sql

In [0]:
def is_select_statement(sql: str) -> bool:
    """
    Verify that the SQL begins with SELECT.

    This intentionally rejects:
        WITH ...
        INSERT ...
        UPDATE ...
        DELETE ...
        CREATE ...
    """

    normalized = sql.strip()

    return bool(
        re.match(
            r"^SELECT\b",
            normalized,
            flags=re.IGNORECASE
        )
    )

In [0]:
def find_forbidden_keywords(sql: str) -> List[str]:
    """
    Return forbidden SQL operations found in the query.
    """

    found = []

    for keyword in FORBIDDEN_KEYWORDS:

        pattern = rf"\b{re.escape(keyword)}\b"

        if re.search(
            pattern,
            sql,
            flags=re.IGNORECASE
        ):
            found.append(keyword)

    return found

In [0]:
def find_forbidden_patterns(sql: str) -> List[str]:
    """
    Detect potentially unsafe functions or external access patterns.
    """

    found = []

    for pattern in FORBIDDEN_PATTERNS:

        if re.search(
            pattern,
            sql,
            flags=re.IGNORECASE
        ):
            found.append(pattern)

    return found

In [0]:
def extract_referenced_tables(sql: str) -> List[str]:
    """
    Extract tables appearing after FROM or JOIN.

    This is a lightweight validator, not a complete SQL parser.
    """

    pattern = r"\b(?:FROM|JOIN)\s+([a-zA-Z0-9_$.]+)"

    matches = re.findall(
        pattern,
        sql,
        flags=re.IGNORECASE
    )

    return list(dict.fromkeys(matches))

In [0]:
def validate_tables(sql: str) -> Dict:

    referenced_tables = extract_referenced_tables(sql)

    unauthorized_tables = [
        table
        for table in referenced_tables
        if table.lower() not in {
            allowed.lower()
            for allowed in ALLOWED_TABLES
        }
    ]

    return {
        "referenced_tables": referenced_tables,
        "unauthorized_tables": unauthorized_tables,
        "valid": len(unauthorized_tables) == 0
    }

In [0]:
def validate_limit(sql: str) -> Dict:
    """
    Validate LIMIT.

    If no LIMIT exists, the validator will require one.
    """

    match = re.search(
        r"\bLIMIT\s+(\d+)",
        sql,
        flags=re.IGNORECASE
    )

    if not match:

        return {
            "valid": False,
            "limit": None,
            "message": "Query must contain a LIMIT clause."
        }

    limit_value = int(match.group(1))

    if limit_value <= 0:

        return {
            "valid": False,
            "limit": limit_value,
            "message": "LIMIT must be greater than zero."
        }

    if limit_value > MAX_RESULT_ROWS:

        return {
            "valid": False,
            "limit": limit_value,
            "message": f"LIMIT cannot exceed {MAX_RESULT_ROWS}."
        }

    return {
        "valid": True,
        "limit": limit_value,
        "message": "LIMIT is valid."
    }

In [0]:
def validate_sql(sql: str) -> Dict:
    """
    Main SQL security validator.

    Returns a structured validation result.
    """

    sql = normalize_sql(sql)

    errors = []

    # --------------------------------------------------------
    # 1. Empty SQL
    # --------------------------------------------------------

    if not sql:

        return {
            "is_valid": False,
            "sql": sql,
            "errors": ["SQL query is empty."],
            "referenced_tables": []
        }

    # --------------------------------------------------------
    # 2. Length
    # --------------------------------------------------------

    if len(sql) > MAX_SQL_LENGTH:

        errors.append(
            f"SQL query exceeds maximum length of {MAX_SQL_LENGTH} characters."
        )

    # --------------------------------------------------------
    # 3. Comments
    # --------------------------------------------------------

    if contains_sql_comments(sql):

        errors.append(
            "SQL comments are not allowed."
        )

    # --------------------------------------------------------
    # 4. Multiple statements
    # --------------------------------------------------------

    if contains_multiple_statements(sql):

        errors.append(
            "Multiple SQL statements are not allowed."
        )

    # --------------------------------------------------------
    # 5. SELECT only
    # --------------------------------------------------------

    if not is_select_statement(sql):

        errors.append(
            "Only SELECT statements are allowed."
        )

    # --------------------------------------------------------
    # 6. Forbidden keywords
    # --------------------------------------------------------

    forbidden_keywords = find_forbidden_keywords(sql)

    if forbidden_keywords:

        errors.append(
            "Forbidden SQL operations detected: "
            + ", ".join(forbidden_keywords)
        )

    # --------------------------------------------------------
    # 7. Forbidden functions / access
    # --------------------------------------------------------

    forbidden_patterns = find_forbidden_patterns(sql)

    if forbidden_patterns:

        errors.append(
            "Forbidden functions or external access patterns detected."
        )

    # --------------------------------------------------------
    # 8. Table allowlist
    # --------------------------------------------------------

    table_validation = validate_tables(sql)

    if not table_validation["valid"]:

        errors.append(
            "Unauthorized table(s): "
            + ", ".join(
                table_validation["unauthorized_tables"]
            )
        )

    # --------------------------------------------------------
    # 9. LIMIT
    # --------------------------------------------------------

    limit_validation = validate_limit(sql)

    if not limit_validation["valid"]:

        errors.append(
            limit_validation["message"]
        )

    # --------------------------------------------------------
    # Final result
    # --------------------------------------------------------

    return {
        "is_valid": len(errors) == 0,
        "sql": sql,
        "errors": errors,
        "referenced_tables": table_validation["referenced_tables"],
        "limit": limit_validation.get("limit")
    }

In [0]:
malicious_sql = """
DROP TABLE genai_copilot.gold.region_sales
"""

validation = validate_sql(malicious_sql)

print(validation)

In [0]:
malicious_sql = """
DELETE FROM genai_copilot.gold.region_sales
"""

validation = validate_sql(malicious_sql)

print(validation)

In [0]:
malicious_sql = """
SELECT *
FROM system.information_schema.tables
LIMIT 10
"""

validation = validate_sql(malicious_sql)

print(validation)

In [0]:
sql_without_limit = """
SELECT
    region,
    SUM(revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
"""

validation = validate_sql(sql_without_limit)

print(validation)

In [0]:
sql_large_limit = """
SELECT *
FROM genai_copilot.gold.region_sales
LIMIT 1000000
"""

validation = validate_sql(sql_large_limit)

print(validation)

In [0]:
multi_statement_sql = """
SELECT *
FROM genai_copilot.gold.region_sales
LIMIT 10;

DROP TABLE genai_copilot.gold.region_sales
"""

validation = validate_sql(multi_statement_sql)

print(validation)

In [0]:
comment_sql = """
SELECT *
FROM genai_copilot.gold.region_sales
-- malicious comment
LIMIT 10
"""

validation = validate_sql(comment_sql)

print(validation)

In [0]:
security_tests = [

    {
        "name": "valid_select",
        "sql": """
        SELECT region, SUM(revenue) AS revenue
        FROM genai_copilot.gold.region_sales
        GROUP BY region
        LIMIT 10
        """,
        "expected": True
    },

    {
        "name": "drop_table",
        "sql": """
        DROP TABLE genai_copilot.gold.region_sales
        """,
        "expected": False
    },

    {
        "name": "delete_data",
        "sql": """
        DELETE FROM genai_copilot.gold.region_sales
        """,
        "expected": False
    },

    {
        "name": "update_data",
        "sql": """
        UPDATE genai_copilot.gold.region_sales
        SET revenue = 0
        """,
        "expected": False
    },

    {
        "name": "unauthorized_table",
        "sql": """
        SELECT *
        FROM system.information_schema.tables
        LIMIT 10
        """,
        "expected": False
    },

    {
        "name": "missing_limit",
        "sql": """
        SELECT *
        FROM genai_copilot.gold.region_sales
        """,
        "expected": False
    },

    {
        "name": "large_limit",
        "sql": """
        SELECT *
        FROM genai_copilot.gold.region_sales
        LIMIT 1000000
        """,
        "expected": False
    },

    {
        "name": "multiple_statements",
        "sql": """
        SELECT *
        FROM genai_copilot.gold.region_sales
        LIMIT 10;

        DROP TABLE genai_copilot.gold.region_sales
        """,
        "expected": False
    }
]

In [0]:
test_results = []

for test in security_tests:

    result = validate_sql(test["sql"])

    passed = result["is_valid"] == test["expected"]

    test_results.append({
        "test_name": test["name"],
        "expected": test["expected"],
        "actual": result["is_valid"],
        "test_status": "PASS" if passed else "FAIL",
        "errors": " | ".join(result["errors"])
    })

for result in test_results:

    print(
        f"{result['test_name']}: "
        f"{result['test_status']}"
    )

In [0]:
failed_tests = sum(
    1
    for result in test_results
    if result["test_status"] == "FAIL"
)

total_tests = len(test_results)

security_status = (
    "PASS"
    if failed_tests == 0
    else "FAIL"
)

print("=" * 60)
print("SQL SECURITY TEST RESULTS")
print("=" * 60)

print(f"Total tests: {total_tests}")
print(f"Failed tests: {failed_tests}")
print(f"Security status: {security_status}")

print("=" * 60)

In [0]:
def approve_sql_for_execution(sql: str) -> str:
    """
    Validate SQL before execution.

    Returns the normalized SQL if valid.

    Raises ValueError if validation fails.
    """

    validation = validate_sql(sql)

    if not validation["is_valid"]:

        error_message = "; ".join(
            validation["errors"]
        )

        raise ValueError(
            f"SQL rejected by security validator: {error_message}"
        )

    return validation["sql"]

In [0]:
safe_sql = """
SELECT
    region,
    SUM(revenue) AS total_revenue
FROM genai_copilot.gold.region_sales
GROUP BY region
ORDER BY total_revenue DESC
LIMIT 10
"""

approved_sql = approve_sql_for_execution(safe_sql)

print("SQL APPROVED")
print("=" * 60)
print(approved_sql)


10.35 Security layers in our project

Our final system will have multiple security layers:

Layer 1 — Prompt

Don't send unnecessary sensitive information to the LLM.

Layer 2 — Schema allowlist

Only expose relevant schemas/tables.

Layer 3 — SQL validator

Reject dangerous SQL.

Layer 4 — Table allowlist

Only Gold analytical tables.

Layer 5 — Read-only execution

Only approved SELECT statements.

Layer 6 — Result limits

Prevent unnecessarily large responses.

Layer 7 — Databricks permissions

Unity Catalog permissions remain the final authorization layer.

The SQL Statement Execution API also enforces permissions on objects used by the statement.

In [0]:
print(security_status)